# SPEC 009 — Build the demo gallery (Kaggle CPU)

Chooses the gallery cases by a fixed rule and crops them. It uses the evaluated predictions from
`kits23-evaluate`, so nothing is predicted again and no GPU is needed.

**Before running**
1. *Add Input*: the outputs of `kits23-stage-1..3`, `kits23-prepare-data` and `kits23-evaluate`.
2. **Save Version → Save & Run All (Commit)**. Expected time: a few minutes.

The output folder `gallery/` is what goes to Hugging Face as a dataset.

The code cells are generated by `scripts/notebooks/build_gallery_notebook.py`.
Edit the repository, not this notebook.

In [ ]:
# Step 0 - folders and packages
import json, os, subprocess, sys
from pathlib import Path
WORK = "/kaggle/working/project23d"
for sub in ['scripts/data', 'scripts/gallery', 'src/project3d', 'src/project3d/data', 'src/project3d/eval']:
    os.makedirs(f"{WORK}/{sub}", exist_ok=True)
def sh(cmd):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
sh([sys.executable, "-m", "pip", "install", "-q", "nibabel"])

Write the project code (generated from the repository).

In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/__init__.py
"""Project_23D: renal CT 3D segmentation."""


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/__init__.py
"""KiTS23 data pipeline (SPEC 002): verify, features, split, nnU-Net export."""


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/manifest.py
"""Locate KiTS23 files and verify them against the pinned checksum manifest.

The manifest is ``data/sources/kits23_v0.1.4_files.tsv`` (D-011): one row per
file with ``case_id, kind, size_bytes, sha256, url``. ``kind`` is ``image`` or
``segmentation``.
"""

from __future__ import annotations

import csv
import hashlib
import re
from dataclasses import dataclass
from pathlib import Path

KINDS = ("image", "segmentation")
CASE_FILE_RE = re.compile(r"^(case_\d{5})\.nii(\.gz)?$")

# Verification statuses. Only OK is a pass; DECOMPRESSED is a warning (the file
# exists but the platform un-gzipped it, so the published hash cannot apply).
OK = "OK"
MISSING = "MISSING"
SIZE_MISMATCH = "SIZE_MISMATCH"
HASH_MISMATCH = "HASH_MISMATCH"
DECOMPRESSED = "DECOMPRESSED_UNVERIFIED"
DUPLICATE = "DUPLICATE"
FAILING = {MISSING, SIZE_MISMATCH, HASH_MISMATCH, DUPLICATE}


@dataclass(frozen=True)
class Expected:
    """One manifest row: the size and SHA-256 a downloaded file must have."""

    case_id: str
    kind: str
    size_bytes: int
    sha256: str


def load_manifest(tsv_path: str | Path) -> dict[tuple[str, str], Expected]:
    """Read the checksum manifest, keyed by ``(case_id, kind)``."""
    out: dict[tuple[str, str], Expected] = {}
    with open(tsv_path, newline="", encoding="utf-8") as fh:
        for row in csv.DictReader(fh, delimiter="\t"):
            if row["kind"] not in KINDS:
                raise ValueError(f"unknown kind {row['kind']!r} for {row['case_id']}")
            key = (row["case_id"], row["kind"])
            if key in out:
                raise ValueError(f"duplicate manifest row {key}")
            out[key] = Expected(row["case_id"], row["kind"], int(row["size_bytes"]), row["sha256"].lower())
    return out


def find_case_files(root: str | Path) -> dict[str, list[Path]]:
    """Recursively find ``case_XXXXX.nii.gz`` (or ``.nii``) files under ``root``.

    Returns every path per case so duplicates can be reported, not silently picked.
    """
    found: dict[str, list[Path]] = {}
    for path in sorted(Path(root).rglob("case_*")):
        m = CASE_FILE_RE.match(path.name)
        if m and path.is_file():
            found.setdefault(m.group(1), []).append(path)
    return found


def sha256_file(path: str | Path, chunk: int = 8 * 1024 * 1024) -> str:
    """SHA-256 of a file, read in chunks so large scans don't fill memory."""
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        while block := fh.read(chunk):
            digest.update(block)
    return digest.hexdigest()


def _check_file(path: Path, expected: Expected, check_hash: bool) -> dict:
    """Status, size and hash of the one file found for a manifest entry."""
    result = {"path": str(path), "status": OK, "size_bytes": path.stat().st_size, "sha256": ""}
    if path.suffix != ".gz":
        result["status"] = DECOMPRESSED
    elif result["size_bytes"] != expected.size_bytes:
        result["status"] = SIZE_MISMATCH
    elif check_hash:
        result["sha256"] = sha256_file(path)
        if result["sha256"] != expected.sha256:
            result["status"] = HASH_MISMATCH
    return result


def verify_kind(
    manifest: dict[tuple[str, str], Expected],
    kind: str,
    root: str | Path,
    check_hash: bool = True,
) -> list[dict]:
    """Check every manifest entry of one kind against the files under ``root``.

    Returns one report row per manifest entry, plus one row per file found
    under ``root`` that the manifest does not list (status ``UNEXPECTED``).
    """
    found = find_case_files(root)
    rows = []
    for (case_id, entry_kind), expected in sorted(manifest.items()):
        if entry_kind != kind:
            continue
        paths = found.pop(case_id, [])
        row = {"case_id": case_id, "kind": kind, "path": "", "status": OK,
               "size_bytes": "", "sha256": ""}
        if not paths:
            row["status"] = MISSING
        elif len(paths) > 1:
            row.update(status=DUPLICATE, path=";".join(str(p) for p in paths))
        else:
            row.update(_check_file(paths[0], expected, check_hash))
        rows.append(row)
    for case_id, paths in sorted(found.items()):
        for path in paths:
            rows.append({"case_id": case_id, "kind": kind, "path": str(path),
                         "status": "UNEXPECTED", "size_bytes": path.stat().st_size, "sha256": ""})
    return rows


def summarize(rows: list[dict]) -> dict[str, int]:
    """Number of report rows per status."""
    counts: dict[str, int] = {}
    for row in rows:
        counts[row["status"]] = counts.get(row["status"], 0) + 1
    return counts


def resolved_paths(rows: list[dict]) -> dict[str, Path]:
    """Map case_id to its single verified path (OK or DECOMPRESSED rows only)."""
    return {r["case_id"]: Path(r["path"]) for r in rows if r["status"] in (OK, DECOMPRESSED)}


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/features.py
"""Per-case label features used to stratify the split (D-012).

Only segmentations are read. Labels: 0 background, 1 kidney, 2 tumor, 3 cyst.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np

VALID_LABELS = {0, 1, 2, 3}
KITS21_LAST_ID = 299  # cases 0-299 are the KiTS21 cohort; 400+ were added in KiTS23


def cohort_of(case_id: str) -> str:
    """Cohort proxy for contrast phase (D-010, D-012); not a phase label."""
    return "kits21" if int(case_id.split("_")[1]) <= KITS21_LAST_ID else "kits23_new"


def label_features(seg: np.ndarray, spacing_mm: tuple[float, ...]) -> dict:
    """Volumes in millilitres and presence flags from a label array."""
    present = {int(v) for v in np.unique(seg)}
    unknown = present - VALID_LABELS
    if unknown:
        raise ValueError(f"unexpected label values {sorted(unknown)}")
    counts = np.bincount(seg.ravel().astype(np.int64), minlength=4)
    voxel_ml = float(np.prod(spacing_mm)) / 1000.0
    return {
        "kidney_ml": round(counts[1] * voxel_ml, 3),
        "tumor_ml": round(counts[2] * voxel_ml, 3),
        "cyst_ml": round(counts[3] * voxel_ml, 3),
        "has_tumor": bool(counts[2] > 0),
        "has_cyst": bool(counts[3] > 0),
    }


def case_features(case_id: str, seg_path: str | Path) -> dict:
    """Read one segmentation and return its features plus header geometry."""
    import nibabel as nib

    img = nib.load(str(seg_path))
    seg = np.asanyarray(img.dataobj)
    if not np.all(np.mod(seg, 1) == 0):
        raise ValueError(f"{case_id}: non-integer label values")
    seg = seg.astype(np.uint8)
    spacing = tuple(float(z) for z in img.header.get_zooms()[:3])
    row = {"case_id": case_id, "cohort": cohort_of(case_id),
           "shape": "x".join(map(str, seg.shape)),
           "spacing_mm": "x".join(f"{s:.4f}" for s in spacing)}
    row.update(label_features(seg, spacing))
    return row


def geometry_mismatch(image_path: str | Path, seg_path: str | Path, atol: float = 1e-3) -> str | None:
    """Compare image and mask headers (no voxel data read). None if they match."""
    import nibabel as nib

    img, seg = nib.load(str(image_path)), nib.load(str(seg_path))
    if img.shape[:3] != seg.shape[:3]:
        return f"shape {img.shape[:3]} vs {seg.shape[:3]}"
    if not np.allclose(img.affine, seg.affine, atol=atol):
        return "affine differs"
    return None


def add_size_bins(rows: list[dict]) -> tuple[float, float]:
    """Add ``size_bin`` in place: tertiles of tumor volume among tumor cases.

    Cases without tumor get ``none``. Returns the two cut points (ml) so they
    can be recorded with the split.
    """
    vols = np.array([r["tumor_ml"] for r in rows if r["has_tumor"]], dtype=float)
    if vols.size == 0:
        cuts = (0.0, 0.0)
    else:
        q1, q2 = np.quantile(vols, [1 / 3, 2 / 3])
        cuts = (float(q1), float(q2))
    for r in rows:
        if not r["has_tumor"]:
            r["size_bin"] = "none"
        elif r["tumor_ml"] <= cuts[0]:
            r["size_bin"] = "small"
        elif r["tumor_ml"] <= cuts[1]:
            r["size_bin"] = "medium"
        else:
            r["size_bin"] = "large"
    return cuts


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/split.py
"""Case-level stratified train/validation/test split (SPEC 002, D-012).

Strata are ``cohort | size_bin | cyst``. Per-stratum counts are allocated by
largest-remainder apportionment, so the global counts hit the target fractions
exactly and every stratum is represented proportionally. Cases are shuffled
within each stratum with a fixed seed.

The saved ``split.json`` is the authority. Its fingerprint is recorded so any
later change is detectable; the split is never regenerated after training.
"""

from __future__ import annotations

import hashlib
import json
import math
import random

ROLES = ("train", "val", "test")


def stratum_key(row: dict) -> str:
    """The stratum a case belongs to, e.g. ``kits23|small|nocyst``."""
    return f"{row['cohort']}|{row['size_bin']}|{'cyst' if row['has_cyst'] else 'nocyst'}"


def _apportion(sizes: dict[str, int], total: int, capacity: dict[str, int]) -> dict[str, int]:
    """Split ``total`` across strata proportionally to ``sizes``, within ``capacity``."""
    n = sum(sizes.values())
    if n == 0 or total == 0:
        return {k: 0 for k in sizes}
    quota = {k: sizes[k] * total / n for k in sizes}
    alloc = {k: min(math.floor(quota[k]), capacity[k]) for k in sizes}
    remaining = total - sum(alloc.values())
    # Largest remainder first; ties broken by stratum name for determinism.
    order = sorted(sizes, key=lambda k: (-(quota[k] - math.floor(quota[k])), k))
    while remaining > 0:
        progressed = False
        for k in order:
            if remaining == 0:
                break
            if alloc[k] < capacity[k]:
                alloc[k] += 1
                remaining -= 1
                progressed = True
        if not progressed:
            raise ValueError("not enough cases to satisfy the requested split")
    return alloc


def make_split(rows: list[dict], seed: int = 42,
               fractions: tuple[float, float, float] = (0.70, 0.15, 0.15)) -> dict[str, str]:
    """Return ``{case_id: role}`` with role in ``train``/``val``/``test``."""
    if abs(sum(fractions) - 1.0) > 1e-9:
        raise ValueError("fractions must sum to 1")
    ids = [r["case_id"] for r in rows]
    if len(set(ids)) != len(ids):
        raise ValueError("duplicate case_id in feature rows")

    strata: dict[str, list[str]] = {}
    for r in rows:
        strata.setdefault(stratum_key(r), []).append(r["case_id"])
    sizes = {k: len(v) for k, v in strata.items()}
    n = len(rows)
    n_test = round(n * fractions[2])
    n_val = round(n * fractions[1])

    test_alloc = _apportion(sizes, n_test, sizes)
    val_alloc = _apportion(sizes, n_val, {k: sizes[k] - test_alloc[k] for k in sizes})

    rng = random.Random(seed)
    roles: dict[str, str] = {}
    for key in sorted(strata):
        members = sorted(strata[key])
        rng.shuffle(members)
        t, v = test_alloc[key], val_alloc[key]
        for i, cid in enumerate(members):
            roles[cid] = "test" if i < t else "val" if i < t + v else "train"
    return dict(sorted(roles.items()))


def fingerprint(roles: dict[str, str]) -> str:
    """SHA-256 of the canonical ``{case_id: role}`` mapping."""
    canon = json.dumps(dict(sorted(roles.items())), sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canon.encode("utf-8")).hexdigest()


def nnunet_splits(roles: dict[str, str]) -> list[dict[str, list[str]]]:
    """nnU-Net ``splits_final.json`` content: one fold, train vs val. Test is excluded."""
    return [{
        "train": sorted(c for c, r in roles.items() if r == "train"),
        "val": sorted(c for c, r in roles.items() if r == "val"),
    }]


def summary(rows: list[dict], roles: dict[str, str]) -> dict:
    """Counts per role, overall and by each stratification variable."""
    out: dict = {"total": {r: 0 for r in ROLES}}
    for var in ("cohort", "size_bin", "has_cyst"):
        out[var] = {}
    for row in rows:
        role = roles[row["case_id"]]
        out["total"][role] += 1
        for var in ("cohort", "size_bin", "has_cyst"):
            bucket = out[var].setdefault(str(row[var]), {r: 0 for r in ROLES})
            bucket[role] += 1
    return out


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/nnunet_export.py
"""Build the nnU-Net v2 raw dataset folder for KiTS23 (SPEC 002).

Mirrors nnU-Net's own ``dataset_conversion/Dataset220_KiTS2023.py``: region-based
labels (kidney+masses, masses, tumor), ``regions_class_order`` (1, 3, 2) and the
``NibabelIOWithReorient`` reader. Two differences, both deliberate:

- only train and validation cases go into ``imagesTr``/``labelsTr``, so the
  locked test cases cannot influence nnU-Net's planning or normalization;
- files are symlinked by default instead of copied, so no extra disk is used.
"""

from __future__ import annotations

import json
import os
import shutil
from pathlib import Path

DATASET_ID = 220
DATASET_NAME = "KiTS2023"
DATASET_JSON = {
    "channel_names": {"0": "CT"},
    "labels": {"background": 0, "kidney": [1, 2, 3], "masses": [2, 3], "tumor": 2},
    "regions_class_order": [1, 3, 2],
    "overwrite_image_reader_writer": "NibabelIOWithReorient",
    "name": DATASET_NAME,
    "release": "0.1.4",
    "reference": "https://github.com/neheller/kits23",
    "licence": "CC BY-NC-SA 4.0 (KiTS23 data terms)",
    "description": "KiTS23 train+val cases only; locked test cases excluded (D-012)",
}


def dataset_folder_name(dataset_id: int = DATASET_ID, name: str = DATASET_NAME) -> str:
    """nnU-Net's folder name for a dataset, e.g. ``Dataset220_KiTS2023``."""
    return f"Dataset{dataset_id:03d}_{name}"


def _file_ending(paths: list[Path]) -> str:
    endings = {".nii.gz" if p.name.endswith(".nii.gz") else p.suffix for p in paths}
    if len(endings) != 1:
        raise ValueError(f"mixed file endings {sorted(endings)}; nnU-Net needs one")
    return endings.pop()


def _place(src: Path, dst: Path, mode: str) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    if mode == "symlink":
        os.symlink(src.resolve(), dst)
    elif mode == "copy":
        shutil.copy2(src, dst)
    else:
        raise ValueError(f"unknown mode {mode!r}")


def build_raw_dataset(roles: dict[str, str], images: dict[str, Path], segs: dict[str, Path],
                      raw_root: str | Path, mode: str = "symlink",
                      dataset_id: int = DATASET_ID) -> Path:
    """Create ``<raw_root>/DatasetXXX_KiTS2023`` and return its path.

    ``imagesTr``/``labelsTr`` hold train+val cases; ``imagesTs`` holds test images.
    Test labels are never placed in the nnU-Net folder.
    """
    trainval = sorted(c for c, r in roles.items() if r in ("train", "val"))
    test = sorted(c for c, r in roles.items() if r == "test")
    missing = [c for c in roles if c not in images or (c in trainval and c not in segs)]
    if missing:
        raise ValueError(f"no verified file for {len(missing)} cases, e.g. {missing[:3]}")

    ending = _file_ending([images[c] for c in roles] + [segs[c] for c in trainval])
    out = Path(raw_root) / dataset_folder_name(dataset_id)
    if out.exists():
        shutil.rmtree(out)
    for c in trainval:
        _place(images[c], out / "imagesTr" / f"{c}_0000{ending}", mode)
        _place(segs[c], out / "labelsTr" / f"{c}{ending}", mode)
    for c in test:
        _place(images[c], out / "imagesTs" / f"{c}_0000{ending}", mode)

    meta = dict(DATASET_JSON, numTraining=len(trainval), file_ending=ending)
    (out / "dataset.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return out


def check_no_test_leak(dataset_dir: str | Path, roles: dict[str, str]) -> None:
    """Fail if any test case appears in imagesTr or labelsTr."""
    test = {c for c, r in roles.items() if r == "test"}
    for sub in ("imagesTr", "labelsTr"):
        for p in (Path(dataset_dir) / sub).iterdir():
            if p.name[:10] in test:
                raise AssertionError(f"test case {p.name[:10]} found in {sub}")


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/eval/__init__.py
"""SPEC 004-005: test-set metrics, statistics and case analysis."""


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/eval/metrics.py
"""Secondary per-case metrics (SPEC 004).

The co-primary Dice and surface Dice come from the official KiTS23 evaluator
(see ``official.py``). This module adds the secondary metrics from
docs/evaluation.md, with the official empty-mask rule for Dice and IoU:
both empty -> 1, exactly one empty -> 0. Precision, recall and HD95 are
undefined (NaN) when their denominator mask is empty; counts are reported.
"""

from __future__ import annotations

import numpy as np
from scipy import ndimage

# Official KiTS hierarchical evaluation classes first, then the remaining single classes.
HECS = {"kidney_and_mass": (1, 2, 3), "mass": (2, 3), "tumor": (2,)}
REGIONS = {**HECS, "kidney": (1,), "cyst": (3,)}
# Copied from kits23.configuration.labels; official.check_config() asserts they still match.
SD_TOLERANCES_MM = {"kidney_and_mass": 1.0330772532390826, "mass": 1.1328796488598762,
                    "tumor": 1.1498198361434828}


def region_mask(seg: np.ndarray, labels: tuple[int, ...]) -> np.ndarray:
    """Boolean mask of the voxels whose label is in ``labels``."""
    return np.isin(seg, labels)


def overlap(pred: np.ndarray, ref: np.ndarray) -> dict:
    """Dice, IoU, precision and recall for two boolean masks."""
    tp = int(np.count_nonzero(pred & ref))
    n_pred, n_ref = int(np.count_nonzero(pred)), int(np.count_nonzero(ref))
    if n_pred == 0 and n_ref == 0:
        dice = iou = 1.0
    elif n_pred == 0 or n_ref == 0:
        dice = iou = 0.0
    else:
        dice = 2 * tp / (n_pred + n_ref)
        iou = tp / (n_pred + n_ref - tp)
    return {"dice": dice, "iou": iou,
            "precision": tp / n_pred if n_pred else float("nan"),
            "recall": tp / n_ref if n_ref else float("nan")}


def _surface(mask: np.ndarray) -> np.ndarray:
    return mask & ~ndimage.binary_erosion(mask, border_value=0)


def hd95(pred: np.ndarray, ref: np.ndarray, spacing_mm: tuple[float, ...]) -> float:
    """95th-percentile symmetric Hausdorff distance in mm (max of the two directions).

    NaN when either mask is empty. Computed inside the bounding box of both
    masks, which holds every surface voxel, so distances are unchanged.
    """
    if not pred.any() or not ref.any():
        return float("nan")
    idx = np.argwhere(pred | ref)
    lo, hi = np.maximum(idx.min(0) - 1, 0), idx.max(0) + 2
    box = tuple(slice(a, b) for a, b in zip(lo, hi, strict=True))
    p = np.pad(pred[box], 1)
    r = np.pad(ref[box], 1)
    sp, sr = _surface(p), _surface(r)
    dist_to_r = ndimage.distance_transform_edt(~sr, sampling=spacing_mm)
    dist_to_p = ndimage.distance_transform_edt(~sp, sampling=spacing_mm)
    return float(max(np.percentile(dist_to_r[sp], 95), np.percentile(dist_to_p[sr], 95)))


def case_metrics(pred_seg: np.ndarray, ref_seg: np.ndarray, spacing_mm: tuple[float, ...]) -> dict:
    """All secondary metrics for one case, flattened as ``<region>_<metric>``."""
    if pred_seg.shape != ref_seg.shape:
        raise ValueError(f"shape {pred_seg.shape} vs {ref_seg.shape}")
    voxel_ml = float(np.prod(spacing_mm)) / 1000.0
    out: dict = {}
    for name, labels in REGIONS.items():
        p, r = region_mask(pred_seg, labels), region_mask(ref_seg, labels)
        for k, v in overlap(p, r).items():
            out[f"{name}_{k}"] = v
        out[f"{name}_hd95_mm"] = hd95(p, r, spacing_mm)
        out[f"{name}_ref_ml"] = round(float(np.count_nonzero(r)) * voxel_ml, 3)
        out[f"{name}_pred_ml"] = round(float(np.count_nonzero(p)) * voxel_ml, 3)
    return out


In [ ]:
%%writefile /kaggle/working/project23d/scripts/data/prepare_kits23.py
"""SPEC 002 pipeline: verify -> features -> split -> nnU-Net raw dataset.

Runs on a Kaggle CPU session (or locally on a small subset). Example::

    python scripts/data/prepare_kits23.py --input-root /kaggle/input \
        --manifest data/sources/kits23_v0.1.4_files.tsv \
        --out-dir /kaggle/working/spec002 --raw-root /tmp/nnUNet_raw

Outputs in ``--out-dir``: verify_report.csv, case_features.csv, split.json,
splits_final.json. The nnU-Net raw dataset is written under ``--raw-root``.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import shutil
import statistics
import sys
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "src"))

from project3d.data import features, manifest, nnunet_export, split

IMAGE_MIN_BYTES = 10 * 1024 * 1024  # CT volumes are tens of MB; label files are ~1 MB


def detect_roots(input_root: Path, min_files: int = 1,
                 link_dir: Path | None = None, link_mode: str = "symlink") -> tuple[Path, Path]:
    """Find the images folder and the segmentations folder by typical file size.

    Segmentation folders with fewer than ``min_files`` case files are ignored (e.g.
    nnU-Net's validation predictions sitting next to the full dataset). Images may
    be split over several folders (one per Kaggle staging shard, D-015); they are
    then linked into ``link_dir/images`` so the pipeline sees one folder.
    """
    by_dir = _file_sizes_by_folder(input_root)
    if not by_dir:
        raise SystemExit(f"no case_XXXXX.nii(.gz) files found under {input_root}")
    big = sorted(d for d, s in by_dir.items() if statistics.median(s) >= IMAGE_MIN_BYTES)
    small = [d for d, s in by_dir.items()
             if statistics.median(s) < IMAGE_MIN_BYTES and len(s) >= min_files]
    n_images = sum(len(by_dir[d]) for d in big)
    if not big or n_images < min_files or len(small) != 1:
        raise SystemExit(f"could not tell images from segmentations; folders found: "
                         f"{ {str(d): len(s) for d, s in by_dir.items()} }. "
                         f"Pass --images-root and --segs-root explicitly.")
    if len(big) == 1:
        return big[0], small[0]
    if link_dir is None:
        raise SystemExit(f"images are split over {len(big)} folders; pass a link folder")
    merged = _link_into_one_folder(big, Path(link_dir) / "images", link_mode)
    print(f"linked {n_images} images from {len(big)} folders into {merged}")
    return merged, small[0]


def _file_sizes_by_folder(input_root: Path) -> dict[Path, list[int]]:
    """Map each folder holding case files to the sizes of those files."""
    by_dir: dict[Path, list[int]] = {}
    for paths in manifest.find_case_files(input_root).values():
        for path in paths:
            by_dir.setdefault(path.parent, []).append(path.stat().st_size)
    return by_dir


def _link_into_one_folder(folders: list[Path], merged: Path, link_mode: str) -> Path:
    """Link (or copy) the case files of several folders into ``merged``; a case found twice is an error."""
    if merged.exists():
        shutil.rmtree(merged)
    for folder in folders:
        for path in sorted(folder.iterdir()):
            if not manifest.CASE_FILE_RE.match(path.name):
                continue
            if (merged / path.name).exists():
                raise SystemExit(f"{path.name} appears in more than one image folder")
            nnunet_export._place(path, merged / path.name, link_mode)
    return merged


def write_csv(path: Path, rows: list[dict]) -> None:
    """Write rows of dicts to a CSV file, using the first row's keys as the header."""
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def build_parser() -> argparse.ArgumentParser:
    """Command-line options for this script."""
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--manifest", type=Path, required=True)
    parser.add_argument("--input-root", type=Path, help="auto-detect images/segmentation folders here")
    parser.add_argument("--images-root", type=Path)
    parser.add_argument("--segs-root", type=Path)
    parser.add_argument("--out-dir", type=Path, required=True)
    parser.add_argument("--raw-root", type=Path, required=True)
    parser.add_argument("--link-mode", choices=("symlink", "copy"), default="symlink")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--subset", action="store_true",
                        help="allow missing cases (local smoke test); never use for the real run")
    parser.add_argument("--skip-hash", action="store_true", help="size check only (faster, weaker)")
    return parser


def main(argv: list[str] | None = None) -> int:
    """Verify the files, measure each case, split once and build the nnU-Net raw dataset."""
    parser = build_parser()
    args = parser.parse_args(argv)

    t0 = time.time()
    args.out_dir.mkdir(parents=True, exist_ok=True)
    if args.images_root and args.segs_root:
        images_root, segs_root = args.images_root, args.segs_root
    elif args.input_root:
        images_root, segs_root = detect_roots(args.input_root, link_dir=args.raw_root.parent / "kits23_merged",
                                              link_mode=args.link_mode)
    else:
        parser.error("give --input-root, or both --images-root and --segs-root")
    print(f"images: {images_root}\nsegmentations: {segs_root}")

    # 1. Verify every file against the pinned manifest.
    exp = manifest.load_manifest(args.manifest)
    report = (manifest.verify_kind(exp, "image", images_root, not args.skip_hash)
              + manifest.verify_kind(exp, "segmentation", segs_root, not args.skip_hash))
    write_csv(args.out_dir / "verify_report.csv", report)
    counts = manifest.summarize(report)
    print(f"[1/4] verify: {counts}  ({time.time() - t0:.0f}s)")
    failing = {s: n for s, n in counts.items() if s in manifest.FAILING or s == "UNEXPECTED"}
    if args.subset:
        failing.pop(manifest.MISSING, None)
    if failing:
        print("STOP: verification failed; see verify_report.csv")
        return 1
    if counts.get(manifest.DECOMPRESSED):
        print("WARNING: some files were un-gzipped by the platform; their hashes cannot be checked")

    images = manifest.resolved_paths([r for r in report if r["kind"] == "image"])
    segs = manifest.resolved_paths([r for r in report if r["kind"] == "segmentation"])
    cases = sorted(set(images) & set(segs))
    if not args.subset and len(cases) != len({c for c, _ in exp}):
        print(f"STOP: only {len(cases)} cases have both image and label")
        return 1

    # 2. Image/mask geometry must agree (SPEC 001 5.4), then label features per case.
    bad = {c: why for c in cases if (why := features.geometry_mismatch(images[c], segs[c]))}
    if bad:
        print(f"STOP: image/mask geometry mismatch: {bad}")
        return 1
    rows = [features.case_features(c, segs[c]) for c in cases]
    cuts = features.add_size_bins(rows)
    write_csv(args.out_dir / "case_features.csv", rows)
    print(f"[2/4] features: {len(rows)} cases, tumor-size tertile cuts (ml) = "
          f"{cuts[0]:.2f}, {cuts[1]:.2f}  ({time.time() - t0:.0f}s)")

    # 3. Split once, fingerprint, save.
    roles = split.make_split(rows, seed=args.seed)
    record = split_record(args.manifest, args.seed, rows, roles, cuts)
    (args.out_dir / "split.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
    (args.out_dir / "splits_final.json").write_text(
        json.dumps(split.nnunet_splits(roles), indent=2), encoding="utf-8")
    print(f"[3/4] split: {record['summary']['total']}  fingerprint {record['fingerprint_sha256'][:16]}...")

    # 4. nnU-Net raw dataset (train+val only in imagesTr).
    ds = nnunet_export.build_raw_dataset(roles, images, segs, args.raw_root, mode=args.link_mode)
    nnunet_export.check_no_test_leak(ds, roles)
    print(f"[4/4] nnU-Net raw dataset: {ds}  ({time.time() - t0:.0f}s total)")
    return 0


def split_record(manifest_path: Path, seed: int, rows: list[dict], roles: dict, cuts: tuple) -> dict:
    """The saved split: settings, fingerprint, per-role summary and the role of every case."""
    return {
        "spec": "SPEC 002",
        "dataset": "KiTS23 v0.1.4",
        # Line endings normalized so Windows and Kaggle copies hash the same.
        "manifest_sha256": hashlib.sha256(
            manifest_path.read_bytes().replace(b"\r\n", b"\n")).hexdigest(),
        "seed": seed,
        "fractions": {"train": 0.70, "val": 0.15, "test": 0.15},
        "strata": "cohort | size_bin | cyst",
        "size_bin_cuts_ml": list(cuts),
        "fingerprint_sha256": split.fingerprint(roles),
        "summary": split.summary(rows, roles),
        "roles": roles,
    }


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/project23d/scripts/gallery/select_cases.py
"""SPEC 009: choose the gallery cases by a fixed rule, so they can't be picked to flatter the model.

Example (Kaggle)::

    python scripts/gallery/select_cases.py --per-case eval_pp/per_case.csv \
        --case-features case_features.csv --out gallery_cases.json

Groups are filled in this order, and a case appears only once:
worst, best, median, small tumor, with cyst. Worst comes first so it can't lose a case.
"""

from __future__ import annotations

import argparse
import csv
import json
import statistics
from pathlib import Path

GROUP_SIZES = {"worst": 2, "best": 2, "median": 3, "small_tumor": 2, "with_cyst": 2}


def _is_true(value: object) -> bool:
    return str(value).strip().lower() in ("1", "true")


def select(per_case: list[dict], features: dict[str, dict]) -> list[dict]:
    """Return ``{"case_id", "group"}`` rows, using ``mean_dice`` and the SPEC 002 features."""
    score = {r["case_id"]: float(r["mean_dice"]) for r in per_case}
    by_score = sorted(score, key=lambda c: (score[c], c))
    chosen: dict[str, str] = {}

    def take(group: str, ordered: list[str]) -> None:
        for case in [c for c in ordered if c not in chosen][:GROUP_SIZES[group]]:
            chosen[case] = group

    def nearest_median(cases: list[str]) -> list[str]:
        mid = statistics.median(score[c] for c in cases)
        return sorted(cases, key=lambda c: (abs(score[c] - mid), c))

    take("worst", by_score)
    take("best", by_score[::-1])
    take("median", nearest_median(by_score))
    take("small_tumor", nearest_median([c for c in by_score if features[c]["size_bin"] == "small"]))
    take("with_cyst", nearest_median([c for c in by_score if _is_true(features[c]["has_cyst"])]))
    return [{"case_id": c, "group": g} for c, g in chosen.items()]


def read_csv(path: Path) -> list[dict]:
    """Read a CSV file into a list of dicts."""
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def main(argv: list[str] | None = None) -> int:
    """Pick the gallery cases and save them as JSON."""
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--per-case", type=Path, required=True)
    parser.add_argument("--case-features", type=Path, required=True)
    parser.add_argument("--out", type=Path, required=True)
    args = parser.parse_args(argv)

    features = {r["case_id"]: r for r in read_csv(args.case_features)}
    rows = select(read_csv(args.per_case), features)
    args.out.write_text(json.dumps(rows, indent=2), encoding="utf-8")
    print(f"selected {len(rows)} cases: {rows}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/project23d/scripts/gallery/build_gallery.py
"""SPEC 009: crop and package the selected test cases for the public gallery.

Example (Kaggle)::

    python scripts/gallery/build_gallery.py --cases gallery_cases.json --per-case eval_pp/per_case.csv \
        --images-root <images> --segs-root <segmentations> --pred-dir pred_pp --out-dir gallery

Uses the evaluated predictions: nothing is predicted again. Each case folder holds ct.nii.gz,
pred.nii.gz, gt.nii.gz and meta.json. ``index.json`` lists all the cases for the website.
"""

from __future__ import annotations

import argparse
import csv
import json
import math
import sys
from pathlib import Path
from typing import TYPE_CHECKING

import numpy as np
import numpy.typing as npt

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "src"))

from project3d.data import features, manifest
from project3d.eval.metrics import REGIONS

if TYPE_CHECKING:
    from nibabel.spatialimages import SpatialImage

MARGIN_MM = 15.0
MAX_CASE_MB = 40  # the worst case is kept whatever its size; a false region makes its crop large
MAX_TOTAL_MB = 300
SCORES = ("mean_dice", "mean_sd", "kidney_and_mass_dice_official", "mass_dice_official", "tumor_dice_official")
VOLUMES = ("kidney", "tumor", "cyst")

DATASET_CARD = """---
license: cc-by-nc-sa-4.0
pretty_name: KiTS23 segmentation demo gallery
tags:
  - medical-imaging
  - ct
  - kidney
  - segmentation
size_categories:
  - n<1K
---

# KiTS23 segmentation demo gallery

> Research demo, not for clinical use.

{n} CT cases from the internal test set of a KiTS23 nnU-Net project, for a public 3D viewer. The
cases were chosen by a fixed rule, and the worst results are included on purpose.

Each case folder holds:

| File | Content |
|---|---|
| `ct.nii.gz` | The CT, cropped |
| `pred.nii.gz` | The model's evaluated prediction, post-processed (0 background, 1 kidney, 2 tumor, 3 cyst) |
| `gt.nii.gz` | The KiTS23 expert annotation |
| `meta.json` | Case ID, selection group, scores and region volumes |

**Crop:** each scan is cropped to the box around the true and predicted foreground, plus {margin:g} mm
on every side. CT, prediction and annotation share one shape and affine. Scores and volumes in
`meta.json` were measured on the full scan.

## Credit and license

These files are derived from KiTS23 v0.1.4 and are shared under CC BY-NC-SA 4.0: non-commercial
use only, shared under the same terms. If you use them, cite KiTS23:

> Heller N, et al. The KiTS21 Challenge: Automatic segmentation of kidneys, renal tumors, and
> renal cysts in corticomedullary-phase CT. arXiv:2307.01984. <https://github.com/neheller/kits23>

The predictions come from nnU-Net v2 (Isensee et al., Nature Methods 2021).
"""


def crop_box(mask: np.ndarray, spacing: tuple, margin_mm: float = MARGIN_MM) -> tuple[slice, ...]:
    """Box around the mask, with a margin in millimetres, kept inside the volume."""
    hits = np.argwhere(mask)
    if hits.size == 0:
        return tuple(slice(0, n) for n in mask.shape)
    box = []
    for axis, size in enumerate(mask.shape):
        pad = math.ceil(margin_mm / float(spacing[axis]))
        box.append(slice(max(int(hits[:, axis].min()) - pad, 0), min(int(hits[:, axis].max()) + 1 + pad, size)))
    return tuple(box)


def _save_crop(img: SpatialImage, box: tuple[slice, ...], dtype: npt.DTypeLike, path: Path) -> None:
    import nibabel as nib

    crop = img.slicer[box]  # updates the affine, so the crop stays in the right place in space
    data = np.round(np.asanyarray(crop.dataobj)).astype(dtype)
    out = nib.Nifti1Image(data, crop.affine)
    out.set_data_dtype(dtype)
    nib.save(out, str(path))


def case_meta(row: dict, group: str, shape: tuple, spacing: tuple) -> dict:
    """Scores and volumes copied from per_case.csv, which comes from the evaluation."""
    def number(key: str) -> float | None:
        value = float(row[key])
        return None if math.isnan(value) else value

    return {"case_id": row["case_id"], "group": group,
            "scores": {key: number(key) for key in SCORES},
            "regions": {name: {"dice": number(f"{name}_dice")} for name in REGIONS},
            "volumes_ml": {name: {"reference": number(f"{name}_ref_ml"), "predicted": number(f"{name}_pred_ml")}
                           for name in VOLUMES},
            "crop_shape": [int(n) for n in shape], "spacing_mm": [float(s) for s in spacing[:3]]}


def build_case(case: dict, row: dict, ct_path: Path, gt_path: Path, pred_path: Path, out_dir: Path) -> dict:
    """Crop the CT, label and prediction around the kidneys and describe the case."""
    import nibabel as nib

    for other in (gt_path, pred_path):
        why = features.geometry_mismatch(ct_path, other)
        if why:
            raise ValueError(f"{case['case_id']}: {other.name} doesn't match the CT ({why})")
    ct, gt, pred = (nib.load(str(p)) for p in (ct_path, gt_path, pred_path))
    spacing = tuple(float(z) for z in ct.header.get_zooms()[:3])
    # the predicted foreground counts too, or a false region outside the kidneys would be hidden
    mask = (np.asanyarray(gt.dataobj) > 0) | (np.asanyarray(pred.dataobj) > 0)
    box = crop_box(mask, spacing)
    folder = out_dir / case["case_id"]
    folder.mkdir(parents=True, exist_ok=True)
    _save_crop(ct, box, np.int16, folder / "ct.nii.gz")
    _save_crop(gt, box, np.uint8, folder / "gt.nii.gz")
    _save_crop(pred, box, np.uint8, folder / "pred.nii.gz")
    meta = case_meta(row, case["group"], tuple(s.stop - s.start for s in box), spacing)
    size_mb = sum(p.stat().st_size for p in folder.glob("*.nii.gz")) / 1e6
    if size_mb > MAX_CASE_MB:
        raise ValueError(f"{case['case_id']} is {size_mb:.1f} MB, over the {MAX_CASE_MB} MB limit")
    meta["size_mb"] = round(size_mb, 2)
    (folder / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return meta


def main(argv: list[str] | None = None) -> int:
    """Build the gallery files for every selected case."""
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--cases", type=Path, required=True)
    parser.add_argument("--per-case", type=Path, required=True)
    parser.add_argument("--images-root", type=Path, required=True)
    parser.add_argument("--segs-root", type=Path, required=True)
    parser.add_argument("--pred-dir", type=Path, required=True)
    parser.add_argument("--out-dir", type=Path, required=True)
    args = parser.parse_args(argv)

    cases = json.loads(args.cases.read_text(encoding="utf-8"))
    with open(args.per_case, newline="", encoding="utf-8") as fh:
        scores = {r["case_id"]: r for r in csv.DictReader(fh)}
    images, segs = manifest.find_case_files(args.images_root), manifest.find_case_files(args.segs_root)
    metas = []
    for case in cases:
        cid = case["case_id"]
        metas.append(build_case(case, scores[cid], images[cid][0], segs[cid][0],
                                args.pred_dir / f"{cid}.nii.gz", args.out_dir))
        print(f"{cid} {case['group']}: {metas[-1]['size_mb']} MB, crop {metas[-1]['crop_shape']}", flush=True)
    total = sum(m["size_mb"] for m in metas)
    if total > MAX_TOTAL_MB:
        print(f"STOP: gallery is {total:.0f} MB, over the {MAX_TOTAL_MB} MB limit")
        return 1
    (args.out_dir / "index.json").write_text(json.dumps(metas, indent=2), encoding="utf-8")
    (args.out_dir / "README.md").write_text(DATASET_CARD.format(n=len(metas), margin=MARGIN_MM), encoding="utf-8",
                                            newline="\n")
    print(f"gallery: {len(metas)} cases, {total:.0f} MB")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
# Step 1 - find the inputs
sys.path[:0] = [f"{WORK}/src", f"{WORK}/scripts/data"]
import prepare_kits23
INPUT = Path("/kaggle/input")
IMAGES, SEGS = prepare_kits23.detect_roots(INPUT, min_files=489, link_dir=Path("/tmp/kits23_merged"))
per_case = sorted(INPUT.rglob("spec004/eval_pp/per_case.csv"))
assert len(per_case) == 1, per_case
SPEC004 = per_case[0].parent.parent
features = sorted(INPUT.rglob("case_features.csv"))
assert len(features) >= 1, "SPEC 002 case_features.csv not found"
OUT = Path("/kaggle/working/gallery")
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Step 2 - select and package
CASES = Path("/kaggle/working/gallery_cases.json")
sh([sys.executable, f"{WORK}/scripts/gallery/select_cases.py", "--per-case", per_case[0],
    "--case-features", features[0], "--out", CASES])
sh([sys.executable, f"{WORK}/scripts/gallery/build_gallery.py", "--cases", CASES, "--per-case", per_case[0],
    "--images-root", IMAGES, "--segs-root", SEGS, "--pred-dir", SPEC004 / "pred_pp", "--out-dir", OUT])
print((OUT / "index.json").read_text())